In [6]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: Mountpoint must not already contain files

In [18]:
import requests
import json
from google.colab import userdata

API_KEY = userdata.get('YANDEX_API_KEY')
FOLDER_ID = userdata.get('YANDEX_FOLDER_ID')

def ask_yandex_gpt(system_prompt, user_prompt, temperature=0.3):
    url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Api-Key {API_KEY}"
    }
    data = {
        "modelUri": f"gpt://{FOLDER_ID}/yandexgpt-lite/latest",
        "completionOptions": {
            "stream": False,
            "temperature": temperature,
            "maxTokens": 2000
        },
        "messages": [
            {"role": "system", "text": system_prompt},
            {"role": "user", "text": user_prompt}
        ]
    }
    response = requests.post(url, headers=headers, json=data)
    return response.json()['result']['alternatives'][0]['message']['text']

In [19]:
# Простые промпты для теста
system_prompt = "Ты — тестовый бот. Отвечай максимально коротко и по делу."
user_prompt = "Привет! Скажи 'Соединение успешно установлено', если ты получил это сообщение."

print("Отправляю тестовый запрос к YandexGPT...")

try:
    # Вызываем нашу функцию
    response = ask_yandex_gpt(system_prompt, user_prompt)

    print("\n✅ Успех! Ответ от модели:")
    print("-" * 30)
    print(response)
    print("-" * 30)

except KeyError as e:
    print(f"\n❌ Ошибка парсинга ответа. Скорее всего, Yandex вернул ошибку авторизации.")
    print("Убедись, что Folder ID скопирован верно и у API-ключа есть права (роль ai.languageModels.user).")
except Exception as e:
    print(f"\n❌ Произошла ошибка: {e}")

Отправляю тестовый запрос к YandexGPT...

✅ Успех! Ответ от модели:
------------------------------
Соединение успешно установлено
------------------------------


In [21]:
import json
import time
import re
import os

drive_path = '/content/drive/MyDrive/research_proposal'
os.makedirs(drive_path, exist_ok=True)
save_file = f'{drive_path}/synthetic_tasks.json'

synthetic_dataset = []

system_prompt_gen = """Ты — эксперт по составлению задач по программированию на Python.
Отвечай СТРОГО в формате JSON, представляющем массив из 5 объектов. Никакого лишнего текста, вводных слов или пояснений.
Формат каждого объекта:
{
  "id": "уникальный строковый номер (например task_1)",
  "instruction": "условие задачи",
  "evaluate_code": "строка с кодом Python функции тестирования (например, def evaluate(solution): ...)",
  "example solution": "строка с кодом правильного решения задачи"]
  "failure_cases": ["строка с кодом неправильного решения 1", "строка с кодом неправильного решения 2"],
}"""


user_prompt_gen = """Сгенерируй 5 задачи по программированию на Python.
Верни только валидный JSON-массив из 5 элементов."""

print("Начинаем генерацию датасета...")

for i in range(10):
    print(f"Итерация {i+1}/50...")

    try:

        response_text = ask_yandex_gpt(system_prompt_gen, user_prompt_gen, temperature=0.6)
        print('response_text')
        cleaned_text = re.sub(r'^```(json)?\s*', '', response_text.strip(), flags=re.IGNORECASE)
        cleaned_text = re.sub(r'\s*```$', '', cleaned_text).strip()

        tasks = json.loads(cleaned_text)
        for idx, task in enumerate(tasks):
            task['id'] = f"task_iter{i+1}_{idx+1}"

        synthetic_dataset.extend(tasks)
        print(f"  Успешно добавлено {len(tasks)} задач. Всего в датасете: {len(synthetic_dataset)}")

    except json.JSONDecodeError:
        print(f"  Ошибка парсинга JSON. Модель вернула невалидный формат. Пропускаем итерацию.")
    except Exception as e:
        print(f"  Возникла ошибка: {e}")

    time.sleep(1)


with open(save_file, 'w', encoding='utf-8') as f:
    json.dump(synthetic_dataset, f, ensure_ascii=False, indent=2)

print("-" * 40)
print(f"Генерация завершена! Успешно собрано задач: {len(synthetic_dataset)}")
print(f"Данные надежно сохранены на Google Drive по пути: {save_file}")

Начинаем генерацию датасета...
Итерация 1/50...
response_text
  Успешно добавлено 5 задач. Всего в датасете: 5
Итерация 2/50...
response_text
  Успешно добавлено 5 задач. Всего в датасете: 10
Итерация 3/50...
response_text
  Успешно добавлено 5 задач. Всего в датасете: 15
Итерация 4/50...
response_text
  Успешно добавлено 5 задач. Всего в датасете: 20
Итерация 5/50...
response_text
  Успешно добавлено 5 задач. Всего в датасете: 25
Итерация 6/50...
response_text
  Успешно добавлено 5 задач. Всего в датасете: 30
Итерация 7/50...
response_text
  Успешно добавлено 5 задач. Всего в датасете: 35
Итерация 8/50...
response_text
  Успешно добавлено 5 задач. Всего в датасете: 40
Итерация 9/50...
response_text
  Успешно добавлено 5 задач. Всего в датасете: 45
Итерация 10/50...
response_text
  Успешно добавлено 5 задач. Всего в датасете: 50
----------------------------------------
Генерация завершена! Успешно собрано задач: 50
Данные надежно сохранены на Google Drive по пути: /content/drive/MyDriv

In [25]:
import requests
import json
from google.colab import userdata

API_KEY = userdata.get('YANDEX_API_KEY')
FOLDER_ID = userdata.get('YANDEX_FOLDER_ID')

def ask_yandex_gpt_arb(system_prompt, user_prompt, temperature=0.3):
    url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Api-Key {API_KEY}"
    }
    data = {
        "modelUri": f"gpt://{FOLDER_ID}/yandexgpt/latest",
        "completionOptions": {
            "stream": False,
            "temperature": temperature,
            "maxTokens": 2000
        },
        "messages": [
            {"role": "system", "text": system_prompt},
            {"role": "user", "text": user_prompt}
        ]
    }
    response = requests.post(url, headers=headers, json=data)
    return response.json()['result']['alternatives'][0]['message']['text']

In [26]:
import json
import time
import re

arbiter_results = []

arbiter_system_prompt = """Ты — строгий LLM-арбитр для ревью задач по программированию.
Твоя цель — найти логические ошибки в сгенерированной задаче.
Оцени задачу по трем критериям:
1. Соответствует ли evaluate_code инструкции? Нет ли там неверных assert'ов (например, 2*3=5)?
2. Покрывают ли failure_cases типичные неправильные решения? Не является ли код в failure_cases на самом деле ПРАВИЛЬНЫМ решением?
3. Нет ли очевидной невыполнимости или противоречий?

Ответь СТРОГО в формате JSON без лишнего текста:
{
  "status": "ok" или "revise",
  "reason_category": "eval_mismatch" | "poor_coverage" | "impossible" | "none",
  "explanation": "Краткое объяснение твоего вердикта на русском языке"
}"""

print("Starting...")

for i, task in enumerate(synthetic_dataset):
    print(f"Проверка задачи {i+1}/{len(synthetic_dataset)} (ID: {task['id']})")

    user_prompt = f"""
    Инструкция: {task['instruction']}
    Код проверки (тесты): {task['evaluate_code']}
    Примеры неправильных решений (failure_cases): {task['failure_cases']}
    """

    try:
        response_text = ask_yandex_gpt_arb(arbiter_system_prompt, user_prompt, temperature=0.1)

        cleaned_text = re.sub(r'^```(json)?\s*', '', response_text.strip(), flags=re.IGNORECASE)
        cleaned_text = re.sub(r'\s*```$', '', cleaned_text).strip()

        evaluation = json.loads(cleaned_text)

        result_record = {
            "task_id": task["id"],
            "instruction": task["instruction"],
            "arbiter_status": evaluation.get("status"),
            "arbiter_reason": evaluation.get("reason_category"),
            "arbiter_explanation": evaluation.get("explanation")
        }
        arbiter_results.append(result_record)
        print(f"Статус: {evaluation.get('status').upper()} | Причина: {evaluation.get('reason_category')}")
        print(f"Объяснение: {evaluation.get('explanation')}")


    except json.JSONDecodeError:
        print(f"Ошибка парсинга ответа Арбитра. Пропускаем.")
    except Exception as e:
        print(f" Ошибка сети или API: {e}")

    time.sleep(1)

arbiter_save_file = f'{drive_path}/arbiter_results.json'
with open(arbiter_save_file, 'w', encoding='utf-8') as f:
    json.dump(arbiter_results, f, ensure_ascii=False, indent=2)

print("Проверка завершена! Результаты успешно сохранены.")

Starting...
Проверка задачи 1/50 (ID: task_iter1_1)
Статус: REVISE | Причина: poor_coverage
Объяснение: Примеры неправильных решений не охватывают все возможные ошибки. Например, функция может возвращать сумму только первых двух элементов, а может возвращать просто первый элемент или всегда константу.
Проверка задачи 2/50 (ID: task_iter1_2)
Статус: OK | Причина: none
Объяснение: Задача сформулирована корректно, тесты соответствуют инструкции, а примеры неправильных решений действительно являются неверными.
Проверка задачи 3/50 (ID: task_iter1_3)
Статус: OK | Причина: none
Объяснение: Задача сформулирована корректно, тесты соответствуют инструкции, а примеры неправильных решений действительно демонстрируют ошибки в понимании задачи.
Проверка задачи 4/50 (ID: task_iter1_4)
Статус: OK | Причина: none
Объяснение: Задача сформулирована корректно, код проверки соответствует инструкции, примеры неправильных решений действительно являются неправильными.
Проверка задачи 5/50 (ID: task_iter1_5)


In [27]:
import json
import pandas as pd


arbiter_save_file = '/content/drive/MyDrive/research_proposal/arbiter_results.json'


with open(arbiter_save_file, 'r', encoding='utf-8') as f:
    results = json.load(f)

df = pd.DataFrame(results)

total_tasks = len(df)
rejected_tasks = len(df[df['arbiter_status'] == 'revise'])
rejection_rate = (rejected_tasks / total_tasks) * 100 if total_tasks > 0 else 0

reasons_dist = df[df['arbiter_status'] == 'revise']['arbiter_reason'].value_counts()
top_reason = reasons_dist.index[0] if not reasons_dist.empty else "N/A"
top_reason_count = reasons_dist.iloc[0] if not reasons_dist.empty else 0
top_reason_pct = (top_reason_count / rejected_tasks) * 100 if rejected_tasks > 0 else 0

print(f"Всего задач проверено: {total_tasks}")
print(f"Отбраковано Арбитром: {rejected_tasks}")
print(f"Rejection Rate: {rejection_rate:.1f}%")

for reason, count in reasons_dist.items():
    pct = (count / rejected_tasks) * 100
    print(f" - {reason}: {count} задач ({pct:.1f}%)")



Всего задач проверено: 50
Отбраковано Арбитром: 6
Rejection Rate: 12.0%
 - poor_coverage: 4 задач (66.7%)
 - eval_mismatch: 2 задач (33.3%)
